# Kronos Walk-Forward Backtest

Walk-forward backtest of Kronos. Runs on a T4. No Turso needed — offline evaluation, no DB writes.

**Secrets** (🔑 left sidebar): `GH_TOKEN` only (a fine-grained GitHub PAT with read access to the private `trades` repo). No `TURSO_DATABASE_URL` or `TURSO_AUTH_TOKEN` required.

**To run:** Runtime → Change runtime type → **T4 GPU**, edit the **CONFIG** cell, then Runtime → **Run all**.

In [ ]:
# Always pull the latest pushed code (fresh clone) + the Kronos model repo.
from google.colab import userdata
gh = userdata.get('GH_TOKEN')
!rm -rf trades && git clone https://{gh}@github.com/tvay11/trades.git
!git clone https://github.com/shiyu-coder/Kronos 2>/dev/null || true
%pip install -q -r trades/forecast/requirements.txt
import os, sys
os.environ['KRONOS_REPO'] = '/content/Kronos'
sys.path.insert(0, '/content/trades/forecast')
%cd /content/trades/forecast

In [ ]:
# ===== CONFIG — edit these =====
TICKERS      = ["NVDA", "AAPL", "MSFT", "AMD", "DASH"]
CADENCE      = 21        # ~monthly cutoffs
HORIZONS     = [5, 10, 20, 60]
SAMPLES      = 30        # lower than production (100) for speed; widens bands slightly
TEMPERATURE  = 0.6
LOOKBACK_DAYS = 1095
OUT_DIR      = "/content/backtest_out"

In [ ]:
from backtest import run_backtest
summary = run_backtest(
    TICKERS, cadence=CADENCE, horizons=HORIZONS, samples=SAMPLES,
    temperature=TEMPERATURE, lookback_days=LOOKBACK_DAYS, out_dir=OUT_DIR,
)

In [ ]:
# Download the CSVs (default). To persist instead, mount Drive and set OUT_DIR there.
from google.colab import files
files.download(f"{OUT_DIR}/backtest_results.csv")
files.download(f"{OUT_DIR}/backtest_summary.csv")

# from google.colab import drive; drive.mount("/content/drive")
# then set OUT_DIR = "/content/drive/MyDrive/kronos_backtest" in the config cell.